# Rebuild the Topic 68 Stance Statements

| Item | Description |
|---|---|
| Goal | Rebuild the text of every legislator statement that T2 placed under Topic 68, consumer data privacy legislation, from the saved speech-turn sentences. The policy summary of T3.2.2 to T3.2.5 starts from this one checked table. |
| Input | `track2_assigned.parquet` (one row per stance record with the cited sentence numbers) and `member_turns.parquet` (the sentences of each legislator turn) from T2.2; `hearings_sst.parquet` from D4 for the hearing dates. |
| Output | `topic68_statements.parquet` in the T3.2 Processed folder: one row per statement with its text, stance, hearing year, and token count. `table_statements.csv` and `table_statements_by_year.csv` under `R - Results/tables/...`. |
| Variables | • `text`: the statement's cited sentences, in the order cited, joined by a space<br>• `hearing_year`: the calendar year of the hearing's first held date<br>• `n_tokens`: tokens of `text` under the bge-large-en-v1.5 tokenizer, start and end tokens included; at most 512 so the statement embeds whole |
| Run | Run 2026-10-31. The table equals the one of the October 2026 run: 374 statements from 119 hearings, 221 support and 153 concern. |


## Step 1. Import packages


In [1]:
import os
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer
from IPython.display import display


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.2 - Topic Side - Policy Summary of the Legislator Statements")
NOTEBOOK_NAME = "T3.2.1 - Statements - Rebuild the Topic 68 Stance Statements"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
T22_PROCESSED = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction" / "Processed"
RECORDS_FILE = T22_PROCESSED / "track2_assigned.parquet"
TURNS_FILE = T22_PROCESSED / "member_turns.parquet"
HEARINGS_FILE = SCRATCH_FOLDER / "D - Data Collection and Processing/D4 - SST Sample Construction" / "Processed" / "hearings_sst.parquet"

TOPIC_ID = 68  # consumer data privacy legislation
ENCODER = "BAAI/bge-large-en-v1.5"  # the encoder every embedding of T3 uses
MAX_TOKENS = 512  # the encoder's input limit
TURN_KEYS = ["packageId", "turn_idx"]
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [3]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Load the Topic 68 stance records

Each record is one statement with a support or concern label and the numbers of the turn sentences it cites. Sentence numbers start at 1.


In [4]:
statements = pd.read_parquet(RECORDS_FILE, filters=[("topic_id", "==", TOPIC_ID)]).rename(columns={"sentences": "sentence_numbers"})
assert statements["stance"].isin(["support", "concern"]).all()
assert statements[TURN_KEYS + ["bioGuideId", "record_idx"]].notna().all().all()
assert not statements.duplicated(TURN_KEYS + ["record_idx"]).any()
print(f"Topic {TOPIC_ID} statements: {len(statements):,}; label: {statements['topic_label'].iloc[0]}")
print(f"By stance: {statements['stance'].value_counts().to_dict()}")
display(statements[TURN_KEYS + ["record_idx", "bioGuideId", "stance", "sentence_numbers"]].head(3))


Topic 68 statements: 374; label: consumer data privacy legislation
By stance: {'support': 221, 'concern': 153}


,packageId,turn_idx,record_idx,bioGuideId,stance,sentence_numbers
0,CHRG-106shrg69984,4,1,K000148,support,[9]
1,CHRG-106shrg69984,9,7,A000356,support,[46]
2,CHRG-106shrg69984,9,8,A000356,support,[48]


## Step 5. Load the speech turns of those hearings

Only the turns of hearings that contain a Topic 68 statement. Each turn carries its sentences as a list.


In [5]:
turns = pd.read_parquet(TURNS_FILE, columns=TURN_KEYS + ["bioGuideId", "sentences"], filters=[("packageId", "in", statements["packageId"].unique().tolist())])
turns = turns.rename(columns={"bioGuideId": "turn_bioGuideId", "sentences": "turn_sentences"})
assert not turns.duplicated(TURN_KEYS).any()
print(f"Speech turns loaded: {len(turns):,} from {turns['packageId'].nunique():,} hearings")


Speech turns loaded: 16,288 from 119 hearings


## Step 6. Attach each statement's turn and rebuild its text

A many-to-one join gives every statement the sentence list of its turn; the speaker on the statement must match the turn's speaker. The cited sentences are taken from the turn in the order cited and joined with a space. Every cited number must fall inside the turn, and no statement may come out empty.


In [6]:
statements = statements.merge(turns, on=TURN_KEYS, how="left", validate="many_to_one")
assert statements["turn_sentences"].notna().all()
assert statements["bioGuideId"].eq(statements["turn_bioGuideId"]).all()
texts = []
for sentence_numbers, turn_sentences in zip(statements["sentence_numbers"], statements["turn_sentences"]):
    sentence_numbers = [int(number) for number in sentence_numbers]
    assert sentence_numbers and min(sentence_numbers) >= 1 and max(sentence_numbers) <= len(turn_sentences)
    texts.append(" ".join(turn_sentences[number - 1] for number in sentence_numbers))
statements["text"] = texts
statements["n_sentences"] = statements["sentence_numbers"].map(len)
assert statements["text"].str.strip().ne("").all()
print(f"Statements with text: {len(statements):,}; sentences per statement: min {statements['n_sentences'].min()}, median {int(statements['n_sentences'].median())}, max {statements['n_sentences'].max()}")
for example in statements.sample(3, random_state=0).itertuples(index=False):
    print(f"\n[{example.stance}] {example.text}")


Statements with text: 374; sentences per statement: min 1, median 1, max 8

[concern] I do have concerns with the risks posed by the EU's General Data Protection Regulations and the California Consumer Privacy Act. Those concern what it will mean to the Internet ecosystem, especially the innovative entrepreneurial businesses that are positioned to be disadvantaged based upon these regulatory overhauls and in some instances these regulations conflict one with another.

[support] Passage of Federal data privacy legislation is long overdue.

[concern] There has to be a way that you can figure out how to do what's necessary-- because we all see people in the airports going through with long skirts and lots of cover-up possibilities, and that's what you're trying to address, and we understand that--there has got to be a way; however, for a privacy concern to be addressed, because it's a legitimate concern.


## Step 7. Attach the hearing year

The hearing's first held date, as the T2.4 figures use, rather than the record's own year field.


In [7]:
hearings = pd.read_parquet(HEARINGS_FILE, columns=["packageId", "heldDate_first"])
assert hearings["packageId"].is_unique
statements = statements.merge(hearings, on="packageId", how="left", validate="many_to_one")
assert statements["heldDate_first"].notna().all()
statements["hearing_year"] = pd.to_datetime(statements["heldDate_first"]).dt.year
print(f"Hearing years: {statements['hearing_year'].min()}-{statements['hearing_year'].max()}; hearings: {statements['packageId'].nunique():,}")


Hearing years: 1999-2025; hearings: 119


## Step 8. Count the tokens of each statement

Every statement must fit the encoder's 512-token limit; otherwise its embedding would be cut off silently.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
statements["n_tokens"] = [len(tokenizer(text)["input_ids"]) for text in statements["text"]]
assert statements["n_tokens"].le(MAX_TOKENS).all()
print(f"Tokens per statement: min {statements['n_tokens'].min()}, median {int(statements['n_tokens'].median())}, max {statements['n_tokens'].max()}; total {statements['n_tokens'].sum():,}")


Tokens per statement: min 8, median 38, max 245; total 17,831


## Step 9. Count statements by stance and hearing year

Coverage for the record, saved as a table.


In [9]:
coverage = pd.crosstab(statements["hearing_year"], statements["stance"])
coverage["total"] = coverage.sum(axis=1)
assert coverage["total"].sum() == len(statements)
coverage.to_csv(TABLE_FOLDER / "table_statements_by_year.csv")
display(coverage.T)


hearing_year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
stance,,,,,,,,,,,,,,,,,,,,,
concern,5,15,7,6,5,2,1,5,1,3,...,11,6,15,9,1,21,0,2,2,3
support,4,18,8,7,3,2,1,3,0,3,...,10,5,29,31,5,41,1,8,7,3
total,9,33,15,13,8,4,2,8,1,6,...,21,11,44,40,6,62,1,10,9,6


## Step 10. Save the statement table

One row per statement in hearing order. Later notebooks read this file and never go back to the turn sentences.


In [10]:
output_columns = ["packageId", "turn_idx", "record_idx", "bioGuideId", "hearing_year", "stance", "text", "n_tokens"]
statements = statements.sort_values(["hearing_year", "packageId", "turn_idx", "record_idx"]).reset_index(drop=True)
statements[output_columns].to_parquet(PROCESSED_FOLDER / "topic68_statements.parquet", index=False)
statements[output_columns].to_csv(TABLE_FOLDER / "table_statements.csv", index=False)
print(f"Saved topic68_statements.parquet and table_statements.csv: {len(statements):,} rows; columns {output_columns}")
display(statements[output_columns].head(3))


Saved topic68_statements.parquet and table_statements.csv: 374 rows; columns ['packageId', 'turn_idx', 'record_idx', 'bioGuideId', 'hearing_year', 'stance', 'text', 'n_tokens']


,packageId,turn_idx,record_idx,bioGuideId,hearing_year,stance,text,n_tokens
0,CHRG-106shrg69984,4,1,K000148,1999,support,The bill is an important first step that recog...,39
1,CHRG-106shrg69984,9,7,A000356,1999,support,"With all due respect, this process is too long...",48
2,CHRG-106shrg69984,9,8,A000356,1999,support,The second area where I think the PROTECT Act ...,48
